# PANDAS MASTERY — CHAPTERS 37–47

```
PANDAS MASTERY
Current Chapters: 37 (Data Cleaning Project), 38 (EDA with Pandas),
                  39 (Feature Engineering with Pandas), 40 (Pandas for ML),
                  41 (Pandas + Scikit-learn), 42 (Data Leakage with Pandas),
                  43 (Pandas Performance), 44 (Large Datasets), 45 (Pandas + SQL),
                  46 (Pandas + NumPy), 47 (Pandas Debugging)
Progress:         47 / 76 chapters
Completed:        Ch1-36
Remaining:        29 chapters
```

Eleven chapters -- everything you've learned, applied for real. A complete messy-dataset
cleaning investigation, a full EDA workflow, feature engineering with explicit leakage-risk
labeling, the Pandas to scikit-learn bridge done correctly (proven, not just claimed), data
leakage made concrete, performance profiling, strategies for larger-than-comfortable data, SQL
and NumPy equivalences, and a full debugging module -- including a genuinely important
current-pandas finding: **Copy-on-Write is now mandatory in pandas 3.0+**, which changes the
entire SettingWithCopyWarning story from what most existing Pandas material teaches. Every
number below was executed and captured before this notebook was written, specifically to avoid
shipping unverified claims.


In [ ]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)


---
# CHAPTER 37 — Data Cleaning Project

## The investigation process, not just the final code
Professional data cleaning is not "run these five functions." It's: inspect, form a hypothesis
about what's wrong, verify the hypothesis, fix it, **re-verify**, move to the next issue. Below
is a deliberately messy synthetic customer dataset, cleaned the way a senior engineer actually
works through one -- narrating each decision.


In [ ]:
raw = pd.DataFrame({
    "customer_id": [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,3,18,19,20],   # 3 appears TWICE (duplicate)
    "name": ["  Alice Smith", "BOB JONES", "carol lee", "Dave Kim", "Eve Chen",
              "Frank Wu", "Grace Lin", None, "Henry Park", "Iris Wong",
              "Jack Ng", "Kate Ho", "Leo Tan", "Mia Chu", "Noah Yip",
              "Olivia Fan", "carol lee ", "Paul Zhao", "Quinn Xu", "Rita Yu"],
    "age": [34, 28, 45, -5, 39, 200, 31, 27, 52, 41, 38, np.nan, 29, 33, 47, 36, 45, 30, 40, np.nan],
    "signup_date": ["2023-01-15", "2023-02-20", "2023/01/30", "2023-03-10", "not_a_date",
                      "2023-02-14", "2023-04-05", "2023-01-22", "2023-03-18", "2023-05-01",
                      "2023-01-10", "2023-06-15", "2023-02-28", "2023-04-20", "2023-03-05",
                      "2023-05-30", "2023-01-30", "2023-07-01", "2023-06-20", "2023-08-10"],
    "revenue": [1250.5, 890.0, 2100.75, 450.25, 1780.0, 3200.5, 670.0, 1100.0, 950.0, 1500.0,
                800.0, 1250.0, 1900.0, 600.0, 2200.0, 1350.0, 2100.75, 750.0, 1600.0, 1050.0],
})
print("raw.shape:", raw.shape)
raw.info()


### Expected Output (verified on pandas 3.0.2)
```
raw.shape: (20, 5)
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  20 non-null     int64
 1   name         19 non-null     str
 2   age          18 non-null     float64
 3   signup_date  20 non-null     str
 4   revenue      20 non-null     float64
dtypes: float64(2), int64(1), str(2)
memory usage: 932.0 bytes
```

### Step 1: `.info()` already flags two issues before looking at values
`name` has 19/20 non-null (one missing), `age` has 18/20 (two missing) -- noted, will handle via
Chapter 12's dropna/fillna toolkit once the *dtype and value* problems are fixed first (fixing
dtype before imputing avoids imputing garbage).

### Step 2: check for duplicate keys -- Chapter 13/36's `nunique()` habit


In [ ]:
print("customer_id unique key check:", raw['customer_id'].nunique(), "distinct vs", len(raw), "rows")
print("Duplicated customer_ids:", raw.loc[raw['customer_id'].duplicated(keep=False), 'customer_id'].tolist())
print(raw[raw['customer_id'].duplicated(keep=False)])


### Expected Output (verified on pandas 3.0.2)
```
customer_id unique key check: 19 distinct vs 20 rows
Duplicated customer_ids: [3, 3]
    customer_id        name   age signup_date  revenue
2             3   carol lee  45.0  2023/01/30  2100.75
16            3  carol lee   45.0  2023-01-30  2100.75
```

### Step 3: investigate -- is this a genuine data error, or a legitimate repeat?
Same `customer_id`, same age, same revenue, nearly identical (whitespace-differing) name, and
the **same date** written in two different formats (`2023/01/30` vs `2023-01-30`) -- this is
extremely strong evidence of a **re-ingestion duplicate** (the same record processed twice,
likely from two different upstream export formats), not two genuinely different events. Per
Chapter 13's lesson, this is exactly the kind of investigation that should happen *before*
blindly dropping -- and here it clearly justifies dropping.

### Step 4: fix dtypes with `errors='coerce'`, THEN handle what that reveals


In [ ]:
clean = raw.drop_duplicates(subset="customer_id", keep="first").copy()

clean["name"] = clean["name"].str.strip().str.title()
clean["signup_date"] = pd.to_datetime(clean["signup_date"], format="mixed", errors="coerce")
clean["age"] = pd.to_numeric(clean["age"], errors="coerce")

print("After dedup:", clean.shape)
print("\nInvalid dates (became NaT):\n", clean[clean["signup_date"].isna()][["customer_id", "signup_date"]])
print("\nImpossible ages:\n", clean[(clean["age"] < 0) | (clean["age"] > 120)][["customer_id", "age"]])


### Expected Output (verified on pandas 3.0.2)
```
After dedup: (19, 5)

Invalid dates (became NaT):
    customer_id signup_date
4             5         NaT

Impossible ages:
    customer_id  age
3             4   -5.0
5             6  200.0
```

### Step 5: convert impossible values to missing, THEN impute -- never impute garbage first


In [ ]:
clean["age"] = np.where((clean["age"] < 0) | (clean["age"] > 120), np.nan, clean["age"])

age_median = clean["age"].median()
clean["age"] = clean["age"].fillna(age_median)

print(f"Age median used for imputation: {age_median}")
print("\nFinal dtypes:\n", clean.dtypes)
print("\nRemaining missing values:\n", clean.isna().sum())
print("\nFinal cleaned dataset:\n", clean)


### Expected Output (verified on pandas 3.0.2)
```
Age median used for imputation: 36.0

Final dtypes:
customer_id             int64
name                      str
age                   float64
signup_date    datetime64[us]
revenue               float64
dtype: object

Remaining missing values:
customer_id    0
name           1
age            0
signup_date    1
revenue        0
dtype: int64

Final cleaned dataset:
    customer_id         name   age signup_date  revenue
0             1  Alice Smith  34.0  2023-01-15  1250.50
1             2    Bob Jones  28.0  2023-02-20   890.00
2             3    Carol Lee  45.0  2023-01-30  2100.75
3             4     Dave Kim  36.0  2023-03-10   450.25
4             5     Eve Chen  39.0         NaT  1780.00
5             6     Frank Wu  36.0  2023-02-14  3200.50
6             7    Grace Lin  31.0  2023-04-05   670.00
7             8          NaN  27.0  2023-01-22  1100.00
8             9   Henry Park  52.0  2023-03-18   950.00
9            10    Iris Wong  41.0  2023-05-01  1500.00
10           11      Jack Ng  38.0  2023-01-10   800.00
11           12      Kate Ho  36.0  2023-06-15  1250.00
12           13      Leo Tan  29.0  2023-02-28  1900.00
13           14      Mia Chu  33.0  2023-04-20   600.00
14           15     Noah Yip  47.0  2023-03-05  2200.00
15           16   Olivia Fan  36.0  2023-05-30  1350.00
17           18    Paul Zhao  30.0  2023-07-01   750.00
18           19     Quinn Xu  40.0  2023-06-20  1600.00
19           20      Rita Yu  36.0  2023-08-10  1050.00
```

### Reading the outcome honestly, including what wasn't fixed
`name` still has 1 missing value, and `signup_date` still has 1 missing (`NaT`, for the row
whose original date was `"not_a_date"`, correctly coerced rather than guessed at). **This is
deliberate, not an oversight** -- a name and a signup date are not safely imputable the way an
age can be (there's no sensible "average name," and a fabricated signup date could corrupt any
downstream time-based analysis) -- the professional choice here is to leave these missing and
flag/investigate them at the business level rather than force an imputation that creates false
confidence. Also notice: the imputed age median (`36.0`) landed on **three different rows**
(customers 4, 6, and 12) -- exactly the kind of repeated, suspiciously-identical value pattern
that's worth documenting so a future analyst doesn't mistake these for three genuinely
coincidental real ages of 36.


---
# CHAPTER 38 — EDA with Pandas

## The workflow
```
Business Question -> Inspection -> Data Quality -> Univariate -> Bivariate -> Group Analysis -> Insights
```
Continuing with `clean` from Chapter 37 -- EDA on genuinely-cleaned data, not raw data, since
garbage-in EDA produces garbage insights.


In [ ]:
print("=== Univariate: revenue distribution ===")
print(clean["revenue"].describe())
print("\nSkew check -- mean vs median:")
print(f"  mean:   {clean['revenue'].mean():.2f}")
print(f"  median: {clean['revenue'].median():.2f}")
print(f"  (mean > median suggests a right-skewed distribution, some high-revenue outlier customers)")


### Expected Output (verified on pandas 3.0.2)
```
=== Univariate: revenue distribution ===
count      19.000000
mean     1336.421053
std       679.882455
min       450.250000
25%       845.000000
50%      1250.000000
75%      1690.000000
max      3200.500000
Name: revenue, dtype: float64

Skew check -- mean vs median:
  mean:   1336.42
  median: 1250.00
  (mean > median suggests a right-skewed distribution, some high-revenue outlier customers)
```


In [ ]:
print("=== Bivariate: does age relate to revenue? ===")
print("Correlation:", round(clean["age"].corr(clean["revenue"]), 3))

clean["age_bracket"] = pd.cut(clean["age"], bins=[0, 30, 40, 50, 100],
                                labels=["<=30", "31-40", "41-50", "50+"])
print("\n=== Group analysis: revenue by age bracket ===")
print(clean.groupby("age_bracket", observed=True)["revenue"].agg(
    avg_revenue="mean", customer_count="count"
))


### Expected Output (verified on pandas 3.0.2)
```
=== Bivariate: does age relate to revenue? ===
Correlation: 0.29

=== Group analysis: revenue by age bracket ===
             avg_revenue  customer_count
age_bracket
<=30         1160.000000               4
31-40        1272.840909              11
41-50        1933.583333               3
50+           950.000000               1
```

### Reading these EDA results as insights, not just numbers
The **weak-to-moderate** linear correlation (`0.29`) undersells what's actually a much more
interesting pattern: the group breakdown shows revenue climbing steadily from the `<=30`
bracket through `31-40` and peaking sharply at `41-50` (nearly double the `<=30` average),
before dropping again at `50+` -- a genuinely **non-linear**, inverted-U-shaped relationship
that a single correlation coefficient (which only measures the strength of a *linear* trend)
substantially understates. Note also the `50+` bracket has only **1 customer** -- its average
(`950.0`) is really just that one customer's revenue, not a statistically meaningful group
average; any real business report should flag small-group statistics like this rather than
presenting them with the same confidence as the 11-customer `31-40` group.

### `pd.cut()` -- binning a continuous variable into meaningful categories (Chapter 20's territory, applied to EDA)
`pd.cut(age, bins=[...], labels=[...])` is the standard tool for exactly this kind of grouped
analysis -- converting a continuous measurement into interpretable buckets for `groupby`,
directly analogous to the NumPy course's `np.digitize`, but returning a proper Pandas
categorical (Chapter 17) with labels attached.

### Interview Insight
> "You compute a correlation of 0.29 between two variables and conclude the relationship is
> weak. What might you be missing?"
Correlation measures only **linear** relationship strength -- two variables can have a strong
**non-linear** relationship (e.g., revenue rising then falling across an age range, as shown
above) while showing only a modest linear correlation. A group-wise breakdown, scatter plot, or
non-linear-relationship test reveals what correlation alone can understate -- and any grouped
statistic should also be checked against its group's sample size before being trusted.


---
# CHAPTER 39 — Feature Engineering with Pandas

## Concept
Every feature below is labeled with its **leakage risk** explicitly -- this discipline (stated,
not assumed) is what Chapter 42 formalizes fully, but it needs to start here, at creation time,
not as an afterthought.


In [ ]:
transactions = pd.DataFrame({
    "customer_id": [1,1,1,2,2,3,3,3,3],
    "date": pd.to_datetime(["2023-01-05","2023-02-10","2023-03-15",
                              "2023-01-20","2023-03-01",
                              "2023-01-10","2023-01-25","2023-02-14","2023-03-20"]),
    "amount": [100, 150, 120, 200, 180, 50, 60, 55, 70],
})

# --- LOW RISK: features computed independently per row, no target/future leakage possible ---
transactions["days_since_prev"] = transactions.groupby("customer_id")["date"].diff().dt.days
transactions["amount_log"] = np.log1p(transactions["amount"])

# --- LOW RISK (if computed correctly): aggregated PAST-ONLY features via shift + expanding ---
transactions["cumulative_spend_before_this_txn"] = (
    transactions.groupby("customer_id")["amount"].transform(lambda s: s.shift(1).expanding().sum())
)
transactions["txn_number"] = transactions.groupby("customer_id").cumcount() + 1

print(transactions)


### Expected Output (verified on pandas 3.0.2)
```
   customer_id       date  amount  days_since_prev  amount_log  cumulative_spend_before_this_txn  txn_number
0            1 2023-01-05     100               NaN    4.615121                                NaN           1
1            1 2023-02-10     150              36.0    5.017280                              100.0           2
2            1 2023-03-15     120              33.0    4.795791                              250.0           3
3            2 2023-01-20     200               NaN    5.303305                                NaN           1
4            2 2023-03-01     180              40.0    5.198497                              200.0           2
5            3 2023-01-10      50               NaN    3.931826                                NaN           1
6            3 2023-01-25      60              15.0    4.110874                               50.0           2
7            3 2023-02-14      55              20.0    4.025352                              110.0           3
8            3 2023-03-20      70              34.0    4.262680                              165.0           4
```

### Why `shift(1)` BEFORE `expanding().sum()` is the entire correctness of that feature
`cumulative_spend_before_this_txn` is meant to answer "how much had this customer spent
**before** this transaction" -- a legitimate, leakage-free feature for predicting something
about *this* transaction (e.g., fraud risk, discount eligibility). Without the `shift(1)`,
`expanding().sum()` alone would include the **current row's own amount** in its own cumulative
total -- technically not "future" leakage, but a subtler **target-adjacent** leakage where the
feature partially encodes the very value being summed for that row. **Always ask: does this
feature's value, for a given row, use only information that would genuinely have been available
before that row's outcome was known?**


In [ ]:
# --- HIGH RISK -- demonstrating a genuine leakage bug, deliberately, for contrast ---
transactions["LEAKY_total_customer_spend"] = transactions.groupby("customer_id")["amount"].transform("sum")

print(transactions[["customer_id", "date", "amount", "cumulative_spend_before_this_txn", "LEAKY_total_customer_spend"]])
print("\n<- LEAKY_total_customer_spend includes ALL of a customer's transactions, including ones that")
print("   happen AFTER the current row's date -- using this to predict anything about an EARLY")
print("   transaction would leak information from transactions that hadn't happened yet.")


### Expected Output (verified on pandas 3.0.2)
```
   customer_id       date  amount  cumulative_spend_before_this_txn  LEAKY_total_customer_spend
0            1 2023-01-05     100                                NaN                          370
1            1 2023-02-10     150                              100.0                          370
2            1 2023-03-15     120                              250.0                          370
3            2 2023-01-20     200                                NaN                          380
4            2 2023-03-01     180                              200.0                          380
5            3 2023-01-10      50                                NaN                          235
6            3 2023-01-25      60                               50.0                          235
7            3 2023-02-14      55                              110.0                          235
8            3 2023-03-20      70                              165.0                          235
```

Row 0 (customer 1's **first** transaction, on Jan 5th) shows `LEAKY_total_customer_spend = 370`
-- the sum of **all three** of that customer's transactions, including two that happen weeks and
months later. A model using this feature to predict something about the January 5th transaction
would be given information from the future -- this is precisely the kind of feature that looks
completely fine in a notebook (no error, plausible-looking number) and only reveals itself as a
bug through unrealistically-good validation metrics, exactly Chapter 42's territory.

### Interview Insight
> "How do you decide whether a group-aggregated feature is safe or leaky?"
Ask, for every single row: could this exact feature value have been computed using **only**
data that existed at or before that row's own timestamp/event? `groupby(...).transform("sum")`
uses the **entire** group, including future rows relative to any given row -- safe only for
features genuinely meant to describe "the whole group" as a static property, never for a
per-transaction feature meant to predict something about that specific transaction's
timing-sensitive outcome.


---
# CHAPTER 40 — Pandas for Machine Learning

## Concept
The exact hand-off point: `X`/`y` separation, then the NumPy course's Chapter 34
`train_test_split_indices` pattern, now applied through Pandas.


In [ ]:
ml_data = pd.DataFrame({
    "age": [34, 28, 45, 39, 52, 31, 27, 41, 38, 29],
    "income": [65000, 48000, 92000, 71000, 105000, 55000, 42000, 78000, 68000, 51000],
    "tenure_months": [12, 6, 36, 24, 48, 8, 4, 30, 18, 10],
    "churned": [0, 1, 0, 0, 0, 1, 1, 0, 0, 1],
})

X = ml_data.drop(columns="churned")
y = ml_data["churned"]

print("X:\n", X)
print("\ntype(X):", type(X).__name__, " type(y):", type(y).__name__)
print("X.shape:", X.shape, " y.shape:", y.shape)


### Expected Output (verified on pandas 3.0.2)
```
X:
    age  income  tenure_months
0   34   65000             12
1   28   48000              6
2   45   92000             36
3   39   71000             24
4   52  105000             48
5   31   55000              8
6   27   42000              4
7   41   78000             30
8   38   68000             18
9   29   51000             10

type(X): DataFrame  type(y): Series
X.shape: (10, 3)  y.shape: (10,)
```

### Why `X.drop(columns="churned")` rather than `X[["age","income","tenure_months"]]`
Both work, but `drop(columns=target)` is more robust to schema changes -- if a new feature
column is added upstream, `drop` automatically includes it in `X` (correct, usually desired
behavior); an explicit column list would silently **exclude** any new column until the list is
manually updated, a common source of "why isn't my new feature being used" bugs in evolving
pipelines.

### Train/test split respecting the NumPy course's index-based pattern


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("X_train.shape:", X_train.shape, " X_test.shape:", X_test.shape)
print("\ny_train distribution:\n", y_train.value_counts())
print("\ny_test distribution:\n", y_test.value_counts())


### Expected Output (verified on pandas 3.0.2)
```
X_train.shape: (7, 3)  X_test.shape: (3, 3)

y_train distribution:
 churned
0    4
1    3
Name: count, dtype: int64

y_test distribution:
 churned
0    2
1    1
Name: count, dtype: int64
```

`stratify=y` kept the churn **rate** roughly consistent between train (`3/7 approx 43%`) and
test (`1/3 approx 33%`) -- close, not identical, since with only 10 total rows perfect
proportional matching isn't always achievable, but it prevented a genuinely unlucky split
(e.g., zero churners landing in the test set at all, which a plain unstratified split could
easily produce on data this small).

### Interview Insight
> "Why prefer `X = df.drop(columns='target')` over explicitly listing feature columns?"
It's robust to upstream schema evolution -- a newly added feature column automatically flows
into `X` without any code change, whereas an explicit column list would silently exclude it
until manually updated. The trade-off: `drop` can accidentally include a column that *shouldn't*
be a feature (e.g., an ID or leaked column) if it's not explicitly excluded -- so this pattern
should be paired with an explicit exclusion list for known non-feature columns, not used
blindly.


---
# CHAPTER 41 — Pandas + Scikit-learn

## Concept
Where should a transformation live -- in Pandas code before scikit-learn ever sees the data, or
inside a scikit-learn `Pipeline`? The answer depends entirely on whether the transformation's
parameters (like a mean for imputation, or category encoding) are learned **from the data** or
not.

## The rule
- **Fit-dependent transformations** (imputation values, scaling parameters, encodings) -- MUST
  live inside a scikit-learn `Pipeline`/transformer, fit only on training data, to avoid Chapter
  42's leakage.
- **Fit-independent transformations** (parsing dates, extracting a day-of-week, computing a
  ratio between two raw columns, string cleaning) -- safe to do in Pandas **before** the
  train/test split, since they don't "learn" anything from the data's distribution.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# --- Safe in Pandas, BEFORE the split: fit-independent transformations ---
ml_data_engineered = ml_data.copy()
ml_data_engineered["income_to_age_ratio"] = ml_data_engineered["income"] / ml_data_engineered["age"]   # pure arithmetic, no fitting

X = ml_data_engineered.drop(columns="churned")
y = ml_data_engineered["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# --- MUST be inside a Pipeline: fit-dependent transformations ---
pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),    # learns the median FROM TRAINING DATA ONLY
    ("scale", StandardScaler()),                        # learns mean/std FROM TRAINING DATA ONLY
    ("model", LogisticRegression()),
])

pipeline.fit(X_train, y_train)
print("Test accuracy:", pipeline.score(X_test, y_test))

# Prove the imputer/scaler parameters came from TRAIN only, not test or the full dataset
imputer_median = pipeline.named_steps["impute"].statistics_
print("\nImputer medians (learned from X_train ONLY):", imputer_median)
print("Actual X_train medians:", X_train.median().tolist())
print("Actual FULL dataset medians (should DIFFER, proving no leakage):", X.median().tolist())


### Expected Output (verified on pandas 3.0.2 / scikit-learn 1.8.0)
```
Test accuracy: 1.0

Imputer medians (learned from X_train ONLY): [3.40000000e+01 6.50000000e+04 1.20000000e+01 1.78947368e+03]
Actual X_train medians: [34.0, 65000.0, 12.0, 1789.4736842105262]
Actual FULL dataset medians (should DIFFER, proving no leakage): [36.0, 66500.0, 15.0, 1804.9932523616735]
```

### Reading this as proof, not just a demonstration
The imputer's learned medians **exactly match** `X_train`'s own medians and **differ** from the
full dataset's medians (age: `34.0` vs `36.0`; income: `65000.0` vs `66500.0`; tenure: `12.0`
vs `15.0`) -- concrete, checkable proof that `SimpleImputer` inside the `Pipeline` only ever saw
training data, never peeked at the test set. This is the actual mechanism that makes `Pipeline`
the professionally-correct tool over manually calling `.fillna(df['col'].median())` on the
**full** dataset before splitting -- that manual approach would silently compute the median from
data including the test set, a real, common, easy-to-miss leakage bug Chapter 42 examines fully.

### Interview Insight
> "Why use a scikit-learn `Pipeline` for imputation/scaling instead of doing it directly in
> Pandas with `.fillna()`/manual standardization?"
A `Pipeline`'s `.fit()` learns transformation parameters (medians, means, stds) **only** from
whatever data it's fit on -- calling `pipeline.fit(X_train, y_train)` guarantees those parameters
never see `X_test`. Doing the equivalent manually in Pandas on the full dataset before splitting
computes that statistic using test data too, silently leaking test-set information into the
training process -- a `Pipeline` makes the correct, split-respecting order structurally enforced
rather than a discipline you have to remember every time.


---
# CHAPTER 42 — Data Leakage with Pandas

## Concept
Chapter 41 showed the *fix*. This chapter shows the *bug*, concretely, for each common leakage
pattern -- wrong approach, why it leaks, correct approach, each demonstrated with real numbers.

## Pattern 1: Imputing with the full dataset's statistic


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# WRONG: compute the fill value from the FULL dataset (includes test rows)
wrong_fill_value = X["age"].median()

# CORRECT: compute the fill value from TRAINING data only
correct_fill_value = X_train["age"].median()

print(f"WRONG (leaky) fill value, from FULL data  : {wrong_fill_value}")
print(f"CORRECT fill value, from TRAIN data only    : {correct_fill_value}")


### Expected Output (verified on pandas 3.0.2)
```
WRONG (leaky) fill value, from FULL data  : 36.0
CORRECT fill value, from TRAIN data only    : 34.0
```

### Reading this honestly
The two medians genuinely differ (`36.0` vs `34.0`) -- using the full-dataset value means the
imputation for training rows is subtly informed by test-set values it should never have seen.
On a small dataset the effect is modest; at real scale, with real distributional differences
between train and test, this kind of leak can measurably inflate validation metrics in a way
that doesn't hold up in production.

## Pattern 2: Group aggregation computed before the split


In [ ]:
customer_txns = pd.DataFrame({
    "customer_id": [1,1,2,2,3,3,4,4],
    "amount": [100, 150, 200, 180, 50, 70, 300, 280],
    "is_fraud": [0, 0, 0, 1, 0, 0, 1, 1],
})

# WRONG: compute each customer's average transaction amount using ALL their transactions,
# THEN split -- every row "knows about" other rows from the SAME customer
customer_txns["WRONG_avg_amount"] = customer_txns.groupby("customer_id")["amount"].transform("mean")

train_idx, test_idx = train_test_split(customer_txns.index, test_size=0.5, random_state=0)
print("Row-level split (WRONG for this grouped feature):")
print("Train customer_ids:", customer_txns.loc[train_idx, "customer_id"].tolist())
print("Test customer_ids :", customer_txns.loc[test_idx, "customer_id"].tolist())


### Expected Output (verified on pandas 3.0.2)
```
Row-level split (WRONG for this grouped feature):
Train customer_ids: [2, 1, 3, 3]
Test customer_ids : [4, 2, 1, 4]
```

Customers `1` and `2` both appear in **both** splits -- their `WRONG_avg_amount` was computed
using all their transactions, some of which now sit in the "other" split. The model implicitly
"sees" test information through this shared, pre-computed feature, regardless of which specific
row it's looking at.

### The correct approach: split by ENTITY first, then compute group features within each split


In [ ]:
unique_customers = customer_txns["customer_id"].unique()
train_customers, test_customers = train_test_split(unique_customers, test_size=0.5, random_state=0)

train_data = customer_txns[customer_txns["customer_id"].isin(train_customers)].copy()
test_data = customer_txns[customer_txns["customer_id"].isin(test_customers)].copy()

train_data["CORRECT_avg_amount"] = train_data.groupby("customer_id")["amount"].transform("mean")
test_data["CORRECT_avg_amount"] = test_data.groupby("customer_id")["amount"].transform("mean")

print("Train customers:", sorted(train_data['customer_id'].unique()))
print("Test customers :", sorted(test_data['customer_id'].unique()))
print("Overlap:", set(train_data['customer_id']) & set(test_data['customer_id']), " (must be empty)")


### Expected Output (verified on pandas 3.0.2)
```
Train customers: [np.int64(1), np.int64(2)]
Test customers : [np.int64(3), np.int64(4)]
Overlap: set()  (must be empty)
```

### Interview Insight
> "You're building a fraud model on transaction-level data where customers have multiple
> transactions. Why might a standard random train/test split (by row) cause leakage?"
If group-level features are computed using all of a customer's transactions, and the split is
done randomly by *row* rather than by *customer*, the same customer's transactions can end up
split across both train and test -- verified above with customers 1 and 2 appearing in both. The
fix is splitting by the **entity** first, ensuring no customer's data appears in both splits,
and only then computing any group-level features separately within each split.


---
# CHAPTER 43 — Pandas Performance

## Concept
Profile, don't guess. This chapter measures the real cost of common inefficiencies --
`apply(axis=1)`, inefficient dtypes -- with actual numbers.


In [ ]:
import timeit

n = 100_000
big = pd.DataFrame({
    "a": np.random.default_rng(0).uniform(0, 100, n),
    "b": np.random.default_rng(1).uniform(0, 100, n),
    "category": np.random.default_rng(2).choice(["X", "Y", "Z"], size=n),
})

t_apply = timeit.timeit(lambda: big.apply(lambda row: row["a"] + row["b"], axis=1), number=1)
t_vectorized = timeit.timeit(lambda: big["a"] + big["b"], number=1)

print(f"apply(axis=1)        : {t_apply:.4f}s")
print(f"Vectorized (a + b)   : {t_vectorized:.4f}s   ({t_apply/t_vectorized:.0f}x faster)")

mem_str = big["category"].memory_usage(deep=True)
mem_cat = big["category"].astype("category").memory_usage(deep=True)
print(f"\ncategory column as str : {mem_str/1024:.1f} KB")
print(f"category column as category: {mem_cat/1024:.1f} KB   ({mem_str/mem_cat:.1f}x smaller)")


### Expected Output (verified on pandas 3.0.2 -- timings vary by machine, magnitude/direction is the point)
```
apply(axis=1)        : 0.5407s
Vectorized (a + b)   : 0.0005s   (1031x faster)

category column as str : 4882.9 KB
category column as category: 97.9 KB   (49.9x smaller)
```

### `df.memory_usage(deep=True)` -- the diagnostic entry point for any "why is this slow/huge" investigation


In [ ]:
print(big.memory_usage(deep=True))
print("\nTotal:", big.memory_usage(deep=True).sum() / 1024, "KB")

optimized = big.copy()
optimized["category"] = optimized["category"].astype("category")
optimized["a"] = optimized["a"].astype("float32")
optimized["b"] = optimized["b"].astype("float32")

print("\nAfter dtype optimization:")
print(optimized.memory_usage(deep=True))
print("\nTotal:", optimized.memory_usage(deep=True).sum() / 1024, "KB")
print(f"\nOverall reduction: {big.memory_usage(deep=True).sum() / optimized.memory_usage(deep=True).sum():.1f}x")


### Expected Output (verified on pandas 3.0.2)
```
Index           132
a            800000
b            800000
category    5000000
dtype: int64

Total: 6445.44140625 KB

Index          132
a           400000
b           400000
category    100150
dtype: int64

Total: 879.181640625 KB

Overall reduction: 7.3x
```

### Interview Insight
> "A colleague's notebook takes 30 minutes to run a feature engineering step on a few hundred
> thousand rows. What would you check first?"
First, whether the slow step uses `.apply(axis=1)` or any per-row Python-level operation where a
vectorized equivalent exists -- verified above at over 1000x slower for simple arithmetic.
Second, `df.memory_usage(deep=True)` to check for inefficient dtypes -- both are checkable in
seconds and routinely explain order-of-magnitude slowdowns without touching algorithm logic.


---
# CHAPTER 44 — Large Datasets

## Concept
When a file is too large to comfortably load in memory: `chunksize` for streaming reads, and
incremental aggregation -- the Pandas-level analog of the NumPy course's Chapter 31 chunked
processing pattern.


In [ ]:
import os, tempfile

tmp_dir = tempfile.mkdtemp()
big_path = os.path.join(tmp_dir, "big_transactions.csv")

n_rows = 500_000
rng = np.random.default_rng(0)
big_data = pd.DataFrame({
    "customer_id": rng.integers(1, 1000, n_rows),
    "amount": rng.uniform(10, 500, n_rows),
})
big_data.to_csv(big_path, index=False)
print(f"Wrote {n_rows:,} rows, file size: {os.path.getsize(big_path)/1024/1024:.1f} MB")

chunk_size = 50_000
total_revenue = 0.0
customer_totals = pd.Series(dtype=float)

for chunk in pd.read_csv(big_path, chunksize=chunk_size):
    total_revenue += chunk["amount"].sum()
    chunk_totals = chunk.groupby("customer_id")["amount"].sum()
    customer_totals = customer_totals.add(chunk_totals, fill_value=0)

print(f"\nTotal revenue (chunked): {total_revenue:,.2f}")

direct_total = pd.read_csv(big_path)["amount"].sum()
print(f"Total revenue (direct load, for verification): {direct_total:,.2f}")
print(f"Match? {np.isclose(total_revenue, direct_total)}")


### Expected Output (verified on pandas 3.0.2)
```
Wrote 500,000 rows, file size: 10.6 MB

Total revenue (chunked): 127,624,034.80
Total revenue (direct load, for verification): 127,624,034.80
Match? True
```

### Why `customer_totals.add(chunk_totals, fill_value=0)` is the correct accumulation pattern
Each chunk only contains a partial view of any given customer's transactions. `.add(...,
fill_value=0)` performs **label-aware** addition (Chapter 1's alignment lesson, now doing real
accumulation work): if `customer_totals` already has an entry for a customer from an earlier
chunk, this chunk's amount adds onto it; if this is the first appearance, `fill_value=0` treats
the "missing" prior total as zero rather than producing `NaN`. Verified above to exactly match
the direct-load total.

### Interview Insight
> "You need to compute the total revenue per customer from a 50GB CSV file that doesn't fit in
> memory. How would you approach this in Pandas?"
`pd.read_csv(path, chunksize=N)` to stream the file in bounded-size pieces, computing a partial
per-customer sum for each chunk, then accumulating those partial sums across chunks with
`.add(..., fill_value=0)` for correct label-aligned accumulation. If the aggregation logic
genuinely needs more than simple chunk-wise accumulation supports, that's the point to consider
moving to Polars, DuckDB, or Spark instead (Chapters 65-67).


---
# CHAPTER 45 — Pandas + SQL

## Concept
Every core SQL clause has a direct Pandas equivalent.

| SQL | Pandas |
|---|---|
| `SELECT col1, col2` | `df[['col1', 'col2']]` |
| `WHERE condition` | `df[df['col'] > value]` or `df.query('col > value')` |
| `GROUP BY col` | `df.groupby('col')` |
| `JOIN ... ON` | `pd.merge(df1, df2, on='key')` |
| `ORDER BY col` | `df.sort_values('col')` |
| `DISTINCT` | `df.drop_duplicates()` |
| `CASE WHEN ... THEN ... END` | `np.where(...)` / `np.select(...)` |
| `HAVING` (filter groups by aggregate) | `.agg(...).query(...)` or `groupby(...).filter(...)` |
| `LIMIT n` | `df.head(n)` |


In [ ]:
employees = pd.DataFrame({
    "id": [1,2,3,4,5,6],
    "department": ["Sales","Eng","Sales","Eng","Marketing","Eng"],
    "salary": [65000,95000,72000,110000,58000,130000],
})

# SQL: SELECT department, AVG(salary) as avg_salary, COUNT(*) as cnt
#      FROM employees GROUP BY department HAVING COUNT(*) >= 2 ORDER BY avg_salary DESC
result = (
    employees
    .groupby("department")
    .agg(avg_salary=("salary", "mean"), cnt=("id", "count"))
    .query("cnt >= 2")
    .sort_values("avg_salary", ascending=False)
)
print("Pandas equivalent of a GROUP BY + HAVING + ORDER BY query:\n", result)


### Expected Output (verified on pandas 3.0.2)
```
Pandas equivalent of a GROUP BY + HAVING + ORDER BY query:
             avg_salary  cnt
department
Eng        111666.666667    3
Sales       68500.000000    2
```

### Line-by-line mapping to the SQL query
- `.groupby("department").agg(avg_salary=..., cnt=...)` -> `GROUP BY department` +
  `SELECT AVG(salary), COUNT(*)`
- `.query("cnt >= 2")` -> `HAVING COUNT(*) >= 2` -- filtering **groups** by an aggregate
  condition, applied *after* aggregation, matching SQL's `HAVING` semantics
- `.sort_values("avg_salary", ascending=False)` -> `ORDER BY avg_salary DESC`

### Interview Insight
> "How would you express a SQL `HAVING` clause in Pandas, and how does it differ from `WHERE`?"
`WHERE` filters individual rows **before** any aggregation. `HAVING` filters **groups** based on
an aggregate condition **after** aggregation -- the Pandas equivalent is `.groupby(...).agg(...)`
followed by `.query(...)` on the aggregated result, or `.groupby(...).filter(...)` operating
directly on groups before aggregating (Chapter 22).


---
# CHAPTER 46 — Pandas + NumPy

## Concept
Since you already know NumPy deeply, this chapter is about the **seams** -- where Pandas hands
off to NumPy, what changes (alignment, missing values, labels) and what doesn't.


In [ ]:
s1 = pd.Series([1, 2, 3], index=["a", "b", "c"])
s2 = pd.Series([10, 20, 30], index=["b", "c", "d"])   # DELIBERATELY different, partially overlapping index

arr1 = s1.to_numpy()
arr2 = np.array([10, 20, 30])   # positionally analogous, but NOT label-aware

print("Pandas alignment-aware addition (s1 + s2):\n", s1 + s2)
print("\nNumPy positional addition (arr1 + arr2):", arr1 + arr2)


### Expected Output (verified on pandas 3.0.2)
```
Pandas alignment-aware addition (s1 + s2):
 a     NaN
b    12.0
c    23.0
d     NaN
dtype: float64

NumPy positional addition (arr1 + arr2): [11 22 33]
```

### This is Chapter 1's alignment lesson, now shown with genuinely different numbers
`s1 + s2` gives `b: 2+10=12`, `c: 3+20=23` (correctly matched by label), and `NaN` for `a`/`d`
(present in only one Series). The raw NumPy arrays add **positionally**: `arr1[0]+arr2[0] =
1+10=11`, a number with no coherent real-world meaning here, since `s1['a']` and `arr2[0]`
don't represent the same entity. **This is precisely why converting to NumPy prematurely can
silently discard the exact safety net -- label alignment -- that made an operation correct.**

### When to stay in Pandas vs drop to NumPy


In [ ]:
df = pd.DataFrame({"a": [1.0, 2.0, 3.0], "b": [4.0, 5.0, 6.0]})

X = df.to_numpy()
print("As NumPy array for a model:\n", X, " dtype:", X.dtype)

df["c"] = df["a"] * df["b"]
print("\nStaying in Pandas preserves row identity through operations:\n", df)


### Expected Output (verified on pandas 3.0.2)
```
As NumPy array for a model:
 [[1. 4.]
 [2. 5.]
 [3. 6.]]  dtype: float64

Staying in Pandas preserves row identity through operations:
     a    b     c
0  1.0  4.0   4.0
1  2.0  5.0  10.0
2  3.0  6.0  18.0
```

### The practical rule
Stay in Pandas as long as you need **row/column identity** to matter -- joining, grouping,
filtering by label. Drop to NumPy (`.to_numpy()`) once the data is a clean, homogeneous
numerical matrix headed for heavy linear algebra or a downstream library expecting a raw array --
the `Pandas -> NumPy -> scikit-learn -> Model` chain from the very first chapter of both courses.

### Interview Insight
> "Why might `pandas_series.to_numpy() + another_series.to_numpy()` give a different result than
> `pandas_series + another_series` directly?"
The Pandas version aligns by **index label** before adding -- mismatched labels produce `NaN`;
the NumPy version adds purely by **position**, silently producing a plausible-looking wrong
result if the two arrays' orderings don't correspond to the same real-world entities.


---
# CHAPTER 47 — Pandas Debugging

## A genuinely important, current finding: Copy-on-Write is now MANDATORY in pandas 3.0+
Before covering the standard error catalog, one structural fact changes everything about how
this chapter is usually taught: **pandas 3.0 makes Copy-on-Write (CoW) permanent -- it can no
longer even be disabled.** This fundamentally changes the classic `SettingWithCopyWarning`
story that essentially every piece of older Pandas material (including most of what exists on
the internet right now) teaches as a live danger.


In [ ]:
df = pd.DataFrame({"age": [25, 35, 45], "salary": [50000, 60000, 70000]})

# The CLASSIC "dangerous chained assignment" pattern that used to require a .loc fix
subset = df[df["age"] > 30]
subset["salary"] = 99999

print("subset after chained assignment:\n", subset)
print("\noriginal df (under mandatory Copy-on-Write, this is UNCHANGED -- no warning even needed):\n", df)


### Expected Output (verified on pandas 3.0.2)
```
subset after chained assignment:
    age  salary
1   35   99999
2   45   99999

original df (under mandatory Copy-on-Write, this is UNCHANGED -- no warning even needed):
    age  salary
0   25   50000
1   35   60000
2   45   70000
```

### What actually changed, precisely
On older Pandas (pre-3.0, without CoW enabled), `df[df["age"] > 30]` might return either a view
or a copy depending on internal implementation details you couldn't easily predict -- the
notorious `SettingWithCopyWarning` existed specifically to flag "you might be mutating a view
you didn't mean to mutate, or this assignment might silently do nothing." **Under pandas 3.0's
mandatory Copy-on-Write, `df[condition]` always behaves as if it returned an independent copy**
-- any subsequent mutation of `subset` provably cannot affect `df`, confirmed above with zero
ambiguity and, notably, with **no warning fired at all**, because there's no longer a genuine
danger to warn about. This is a real, structural improvement -- but it also means any advice you
find explaining `SettingWithCopyWarning` as an active danger to code around is describing
pre-3.0 behavior, not what you'll actually see today.

### The `.loc[]` pattern is now about **clarity**, not correctness


In [ ]:
df2 = pd.DataFrame({"age": [25, 35, 45], "salary": [50000, 60000, 70000]})
df2.loc[df2["age"] > 30, "salary"] = 99999
print(".loc[] approach (still the clearer, more explicit way to express intent):\n", df2)


### Expected Output (verified on pandas 3.0.2)
```
.loc[] approach (still the clearer, more explicit way to express intent):
    age  salary
0   25   50000
1   35   99999
2   45   99999
```

Both approaches now produce a correct, isolated result under mandatory CoW -- but `.loc[]` still
reads as a single, clear, atomic "update these specific cells" statement rather than two
separate steps (select, then mutate), and remains the recommended style for that clarity, even
though the historical *correctness* reason for preferring it no longer applies as sharply as it
used to.

### Interview Insight
> "Explain `SettingWithCopyWarning` and how you'd fix it."
A strong, **current** answer should lead with: "On pandas 3.0+, this is largely a legacy
concern -- Copy-on-Write is now mandatory and can't be disabled, so chained assignment like
`df[cond]['col'] = value` is guaranteed not to silently fail to mutate the original or corrupt
it unexpectedly; there's no warning because there's no longer a real danger." A candidate should
still know the historical mechanism (ambiguous view-vs-copy semantics on pre-3.0 Pandas, and
`.loc[]` as the traditional single-step fix), but presenting it as an active, current danger
signals outdated knowledge if the target environment is on a modern Pandas version -- always
verify which Pandas version a specific role/codebase actually runs before assuming this
matters operationally.

## The rest of the standard error catalog


In [ ]:
# KeyError -- accessing a column/label that doesn't exist
try:
    df["nonexistent_column"]
except KeyError as e:
    print("KeyError:", e)

# Merge row-explosion (Chapter 25's lesson, reproduced here as a debugging catalog entry)
left = pd.DataFrame({"key": ["A", "A", "B"], "val": [1, 2, 3]})
right = pd.DataFrame({"key": ["A", "A", "B"], "val2": [10, 20, 30]})
merged = pd.merge(left, right, on="key")
print("\nMerge row-explosion: left has", len(left), "rows, right has", len(right),
      "rows, merged has", len(merged), "rows")
print(merged)


### Expected Output (verified on pandas 3.0.2)
```
KeyError: 'nonexistent_column'

Merge row-explosion: left has 3 rows, right has 3 rows, merged has 5 rows
  key  val  val2
0   A    1    10
1   A    1    20
2   A    2    10
3   A    2    20
4   B    3    30
```

This is Chapter 25's `m x n` rule in miniature: key "A" appears twice on both sides, producing
`2x2=4` rows just for "A"; key "B" appears once on each side, producing `1x1=1` row -- total `5`,
not `3` as a naive glance at the input sizes might suggest.

### Merging on mismatched dtypes -- now raises an error rather than silently misbehaving


In [ ]:
left_typed = pd.DataFrame({"id": [1, 2, 3]})              # id is int64
right_typed = pd.DataFrame({"id": ["1", "2", "3"], "val": [10, 20, 30]})   # id is str

print("left dtypes:\n", left_typed.dtypes)
print("\nright dtypes:\n", right_typed.dtypes)

try:
    result = pd.merge(left_typed, right_typed, on="id")
    print("\nMerge succeeded, rows:", len(result))
except ValueError as e:
    print("\nMerge RAISED an error instead of silently producing zero/wrong matches:")
    print(" ", e)


### Expected Output (verified on pandas 3.0.2)
```
left dtypes:
 id    int64
dtype: object

right dtypes:
 id       str
val    int64
dtype: object

Merge RAISED an error instead of silently producing zero/wrong matches:
  You are trying to merge on int64 and str columns for key 'id'. If you wish to proceed you should use pd.concat
```

### Why this is a genuine, welcome current improvement worth knowing about explicitly
On older Pandas, merging `int64` against `str`/`object` id columns could silently produce **zero
matches** (since `1 != "1"` as values) without any error at all -- the merge would "succeed,"
return an empty-looking result, and the root cause (a dtype mismatch, not a genuine absence of
matching records) could be genuinely difficult to spot, especially in a larger pipeline where
the empty result isn't immediately obvious. **Pandas 3.0 now raises a clear `ValueError`
immediately**, naming exactly which column and which two dtypes are in conflict -- this is
precisely the kind of "loud failure instead of silent wrong result" improvement that the NumPy
course's numerical-edge-cases material would recognize as good defensive design, now built into
Pandas itself rather than requiring `validate=` to catch it after the fact.

### Interview Insight
> "You merge two tables on an ID column and get zero matching rows, even though you're certain
> matching IDs exist in both tables. What would you check first?"
The dtype of the join key in each table -- a very common cause is one side storing the ID as
`int64` and the other as `str`/`object` (e.g., one table loaded from a system that quotes IDs,
the other from one that doesn't). On current Pandas (3.0+), this now raises an explicit
`ValueError` naming the conflicting dtypes rather than silently returning zero matches, which
directly points you at the fix (`astype()` one side to match the other) rather than requiring
you to debug an empty result from scratch.


---
## Chapters 37–47 Summary

- **Data cleaning project**: dedup investigation before deletion, dtype-fix-then-impute
  ordering, and the deliberate choice to leave some values missing rather than force an
  unjustifiable imputation.
- **EDA**: a `0.29` correlation understated a real inverted-U-shaped group pattern (verified) --
  always pair correlation with group-wise breakdowns, and flag small-sample group statistics.
- **Feature engineering**: every feature should be explicitly leakage-labeled at creation time;
  `shift(1)` before `expanding()`/`transform("sum")` is the difference between a safe
  past-only feature and a leaky one.
- **Pandas for ML**: `X = df.drop(columns=target)` is schema-robust; `stratify=y` keeps class
  balance consistent across splits.
- **Pandas + scikit-learn**: verified live that a `Pipeline`'s learned imputation statistics
  exactly match the training split and differ from the full dataset.
- **Data leakage**: full-dataset imputation statistics measurably differ from train-only
  statistics (36.0 vs 34.0, verified); group features computed before an entity-level split can
  leak when the same entity's rows land on both sides (customers 1 and 2 verified appearing in
  both splits under row-level splitting) -- fixed by splitting on the entity first.
- **Performance**: `apply(axis=1)` measured over 1000x slower than vectorized arithmetic;
  category dtype gave a verified ~50x memory reduction on a repetitive text column.
- **Large datasets**: `chunksize` + `.add(..., fill_value=0)` verified to exactly match a
  direct full-load computation.
- **Pandas + SQL**: `HAVING` maps to post-aggregation filtering, distinct from `WHERE`'s
  pre-aggregation row filtering.
- **Pandas + NumPy**: the same "addition" gives genuinely different results in Pandas
  (label-aligned) vs NumPy (positional) -- verified side by side.
- **Debugging**: pandas 3.0's mandatory Copy-on-Write means the classic
  `SettingWithCopyWarning` danger is now structurally eliminated, not just warned about --
  verified live with zero warnings and a provably unaffected original DataFrame; merging on
  mismatched dtypes now raises a clear `ValueError` instead of silently returning zero matches.

---

## Mastery Exercises

**Exercise 1:** In the Chapter 37 cleaning project, `age` was imputed but `name` and
`signup_date` were left missing. Write one sentence justifying each of those three decisions
differently.

**Exercise 2 (Prediction):** For two Series with completely non-overlapping indices, predict
what `s1 + s2` produces, using Chapter 1/46's alignment rule.

**Coding Challenge:** Extend Chapter 44's chunked processing to also compute, per customer, the
**maximum** single transaction amount across all chunks -- requiring a different accumulation
strategy than simple addition, since max isn't naturally incrementally-summable via `.add()`.

**Debugging Challenge:**
```python
df["customer_avg_spend"] = df.groupby("customer_id")["amount"].transform("mean")
X_train, X_test = train_test_split(df, test_size=0.2, random_state=0)
model.fit(X_train.drop(columns="target"), X_train["target"])
```
A teammate is confused why validation performance looks great but production performance is
much worse. Identify the leakage pattern from Chapter 42 at play here, and the fix.

**Interview Questions:**
1. Why should dtype/value cleaning happen before missing-value imputation, not after?
2. Why is entity-level (not row-level) splitting required when group-aggregated features are
   involved?
3. Explain why `SettingWithCopyWarning` is largely a legacy concern on pandas 3.0+, and what
   structural change caused that.
4. Translate `SELECT dept, COUNT(*) FROM employees GROUP BY dept HAVING COUNT(*) > 5` into
   Pandas.

**Quiz:**
- Q1: True/False -- `df.drop(columns='target')` will automatically include any new feature
  column added to the DataFrame later, without code changes.
- Q2: What does `.add(other, fill_value=0)` do differently from plain `+` when two Series have
  non-overlapping labels?
- Q3: On pandas 3.0+, what happens when you attempt to merge an `int64` column against a `str`
  column on the same key name?

---

 
